# Lesson 3: pandas

**Week 3 · Data Engineering Course**

---

**What you will learn:**
- What a DataFrame is and how it differs from a CSV or a list
- Loading, exploring, and understanding a dataset
- Selecting columns and filtering rows
- Cleaning: whitespace, casing, types, missing values
- Grouping and aggregating data
- Writing a cleaned DataFrame to CSV

**Before starting:** install pandas if you have not yet:
```
pip install pandas
```

In [ ]:
import pandas as pd
print(f'pandas version: {pd.__version__}')

---

## 3.1 DataFrames and Series

A **DataFrame** is a table: rows and columns, like a spreadsheet or SQL table. Each column is a **Series** — a 1D labelled array.

pandas keeps the data in memory and lets you query, filter, and transform it in Python.

In [ ]:
# Create a small DataFrame from a dict
data = {
    'product': ['Laptop', 'Mouse', 'Keyboard'],
    'category': ['Electronics', 'Electronics', 'Electronics'],
    'price': [1200.00, 35.00, 75.00]
}

df = pd.DataFrame(data)
print(df)
print()
print(f'Type: {type(df)}')
print(f'Shape: {df.shape}')  # (rows, columns)
print(f'Columns: {list(df.columns)}')

---

## 3.2 Reading CSV Files

In [ ]:
df = pd.read_csv('../data/raw/sales_q1.csv')
print(f'Shape: {df.shape}')   # (rows, columns)
print(f'Columns: {list(df.columns)}')

---

## 3.3 Exploring the Data

Always explore before you clean. These four methods give you an instant picture of any dataset.

In [ ]:
df.head(5)   # first 5 rows

In [ ]:
df.info()    # column names, non-null counts, data types

In [ ]:
df.describe()  # statistics for numeric columns

In [ ]:
# Count unique values in a column
print(df['category'].value_counts())
print()
print(f'Null values per column:')
print(df.isnull().sum())

---

## 3.4 Selecting Columns and Filtering Rows

In [ ]:
# Select one column -> returns a Series
print(df['category'].unique())

# Select multiple columns -> returns a DataFrame
subset = df[['order_id', 'customer_name', 'product', 'total']]
print(subset.head(3))

In [ ]:
# Filter rows using a boolean condition
electronics = df[df['category'].str.strip().str.upper() == 'ELECTRONICS']
print(f'Electronics rows: {len(electronics)}')
print(electronics[['order_id', 'product', 'category']].head())

In [ ]:
# Multiple conditions with & (and) and | (or)
# Parentheses are REQUIRED around each condition
high_value_electronics = df[
    (df['category'].str.strip().str.upper() == 'ELECTRONICS') &
    (pd.to_numeric(df['unit_price'].astype(str).str.replace('$', ''), errors='coerce') > 100)
]
print(high_value_electronics[['order_id', 'product', 'unit_price']].head())

---

## 3.5 Cleaning Data

This is what pandas is most used for in data engineering. Each step below fixes one type of problem.

In [ ]:
# Work on a copy so the original is preserved
clean = df.copy()

# --- 1. Remove duplicate rows ---
before = len(clean)
clean = clean.drop_duplicates(subset=['order_id'])
print(f'Duplicates removed: {before - len(clean)}')

# --- 2. Strip whitespace from all string columns ---
str_cols = clean.select_dtypes(include='object').columns
for col in str_cols:
    clean[col] = clean[col].str.strip()
print(f'Whitespace stripped from: {list(str_cols)}')

In [ ]:
# --- 3. Standardise category casing ---
print('Before:', clean['category'].value_counts().to_dict())
clean['category'] = clean['category'].str.strip().str.title()

# Fix specific typos after normalising
clean['category'] = clean['category'].replace({
    'Home & Kitchen': 'Home & Kitchen',  # already correct
    'Electronics': 'Electronics',
    'Clothing': 'Clothing'
})
print('After:', clean['category'].value_counts().to_dict())

In [ ]:
# --- 4. Clean unit_price: remove $ and convert to float ---
clean['unit_price'] = (
    clean['unit_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
clean['unit_price'] = pd.to_numeric(clean['unit_price'], errors='coerce')
print(clean['unit_price'].dtype)        # float64
print(clean['unit_price'].isnull().sum())# 0

In [ ]:
# --- 5. Fill missing total values ---
clean['total'] = pd.to_numeric(clean['total'], errors='coerce')
missing_total = clean['total'].isnull().sum()
print(f'Missing totals before: {missing_total}')

clean['total'] = clean['total'].fillna(
    clean['quantity'] * clean['unit_price']
)
print(f'Missing totals after: {clean["total"].isnull().sum()}')

In [ ]:
# --- 6. Drop invalid rows (negative or zero quantity) ---
clean['quantity'] = pd.to_numeric(clean['quantity'], errors='coerce')
invalid = clean[clean['quantity'] <= 0]
print(f'Invalid quantity rows: {len(invalid)}')
print(invalid[['order_id', 'customer_name', 'product', 'quantity']])

clean = clean[clean['quantity'] > 0]
print(f'Rows remaining: {len(clean)}')

---

## 3.6 Groupby and Aggregation

In [ ]:
# Revenue by category
revenue_by_cat = (
    clean
    .groupby('category')['total']
    .agg(orders='count', total_revenue='sum')
    .sort_values('total_revenue', ascending=False)
)
print(revenue_by_cat)

# Multiple columns
summary = (
    clean
    .groupby(['category', 'region'])
    .agg(
        orders=('order_id', 'count'),
        revenue=('total', 'sum'),
        avg_order=('total', 'mean')
    )
    .round(2)
    .sort_values('revenue', ascending=False)
)
print(summary.head(8))

---

## 3.7 Writing to CSV

In [ ]:
from pathlib import Path

out_dir = Path('../data/clean')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / 'sales_q1_clean.csv'
clean.to_csv(out_path, index=False)  # index=False: don't write the row numbers

print(f'Saved {len(clean)} rows to {out_path}')

# Verify by reading back
verify = pd.read_csv(out_path)
print(f'Re-read: {verify.shape}')

---

## 3.8 Key Takeaways

1. `pd.read_csv()` loads a CSV into a DataFrame — the starting point for most pandas work.
2. Always explore first: `.head()`, `.info()`, `.describe()`, `.value_counts()`, `.isnull().sum()`.
3. Filter rows with boolean conditions in `[]`. Use `&` for AND and `|` for OR. **Always parenthesise each condition.**
4. `.str.strip()` / `.str.lower()` / `.str.title()` work on a whole column at once — no loop needed.
5. `pd.to_numeric(..., errors='coerce')` converts a column to numbers and turns unparseable values into NaN rather than crashing.
6. `.drop_duplicates(subset=[...])` removes rows where the specified columns repeat.
7. `.fillna(...)` replaces NaN — use it to compute missing totals from other columns.
8. `.groupby().agg()` is the pandas equivalent of SQL `GROUP BY` with aggregate functions.
9. Always save with `index=False` so pandas does not write a spurious row-number column.